# Listing Week 6

In [1]:
import pandas as pd 

listing = pd.read_csv('../../IDX_Data/listing_cleaned_week4_5.csv')

print('Shape:', listing.shape)
listing.head()

/var/folders/hx/s9px6scd2pl5dqfp0pldbdgc0000gn/T/ipykernel_95293/716581094.py:3: DtypeWarning: Columns (0: ListAgentEmail, 1: BuyerAgencyCompensationType) have mixed types. Specify dtype option on import or set low_memory=False.
  listing = pd.read_csv('../../IDX_Data/listing_cleaned_week4_5.csv')


Shape: (539597, 76)


,OriginalListPrice,ListingKey,ListAgentEmail,CloseDate,ClosePrice,ListAgentFirstName,ListAgentLastName,Latitude,Longitude,UnparsedAddress,...,PostalCode,BuyerOfficeName.1,AssociationFee,LotSizeSquareFeet,UnparsedAddress.1,year_month,rate_30yr_fixed,listing_after_close_flag,purchase_after_close_flag,negative_timeline_flag
0,1340000.0,1074973329,haleh360@Gmail.com,NaN,NaN,Haleh,Dowlatshahi,34.052207,-118.408445,2220 Avenue Of The Stars 2704,...,90067,NaN,2105.00,177861.0,2220 Avenue Of The Stars 2704,2024-01,6.6425,False,False,False
1,2500000.0,1074954552,Reneechen@yourhomesoldguaranteed.com,NaN,NaN,Renee,Chen,33.496363,-117.691677,16 Palisades,...,92677,NaN,254.00,5300.0,16 Palisades,2024-01,6.6425,False,False,False
2,3150000.0,1074936537,anader@dppre.com,NaN,NaN,Margaret,Nader,34.119345,-118.111254,1615 Waverly Road,...,91108,NaN,NaN,9404.0,1615 Waverly Road,2024-01,6.6425,False,False,False
3,3090000.0,1074917818,QIANYU0607@GMAIL.COM,NaN,NaN,QIANYU,GUAN,33.984057,-117.802819,2250 Indian Creek Road,...,91765,NaN,295.95,58232.0,2250 Indian Creek Road,2024-01,6.6425,False,False,False
4,12725000.0,1074143166,jeff.williams@pacificsir.com,NaN,NaN,Jeff,Williams,33.607583,-117.887743,317 E. Bayfront,...,92662,NaN,0.00,2250.0,317 E. Bayfront,2024-01,6.6425,False,False,False


In [3]:
print('ListPrice <= 0:', (listing['ListPrice'] <= 0).sum())
print('LivingArea <= 0:', (listing['LivingArea'] <= 0).sum())
print('DaysOnMarket < 0:', (listing['DaysOnMarket'] < 0).sum())

print('\nDate types:')
print(listing[['ListingContractDate']].dtypes)

ListPrice <= 0: 0
LivingArea <= 0: 0
DaysOnMarket < 0: 0

Date types:
ListingContractDate    str
dtype: object


In [4]:
# Convert ListingContractDate to datetime
listing['ListingContractDate'] = pd.to_datetime(
    listing['ListingContractDate'],
    errors='coerce'
)
# Verify
print(listing['ListingContractDate'].dtype)

datetime64[us]


In [5]:
# --- Feature Engineering ---

# Price per square foot
listing['price_per_sqft'] = listing['ListPrice'] / listing['LivingArea']

# Year-month feature (for time analysis)
listing['year_month'] = listing['ListingContractDate'].dt.to_period('M')

# Quick check
listing[['ListPrice', 'LivingArea', 'price_per_sqft', 'year_month']].head()

,ListPrice,LivingArea,price_per_sqft,year_month
0,1340000.0,1301.0,1029.976941,2024-01
1,2500000.0,2788.0,896.700143,2024-01
2,3150000.0,3250.0,969.230769,2024-01
3,3090000.0,7456.0,414.431330,2024-01
4,12725000.0,2321.0,5482.550625,2024-01


In [7]:
# --- Remove extreme price_per_sqft outliers ---

before_rows = listing.shape[0]

listing = listing[
    (listing['price_per_sqft'] > 50) & 
    (listing['price_per_sqft'] < 2000)
]

after_rows = listing.shape[0]

print('Rows before filtering:', before_rows)
print('Rows after filtering:', after_rows)

Rows before filtering: 539597
Rows after filtering: 532739


In [8]:
listing['price_per_sqft'].describe()

count    532739.000000
mean        599.801284
std         309.403060
min          50.189394
25%         372.961778
50%         543.933054
75%         744.641193
max        1999.999333
Name: price_per_sqft, dtype: float64

In [13]:
county_summary = (
    listing
    .groupby('CountyOrParish')
    .agg({
        'ListPrice': ['mean', 'median'],
        'price_per_sqft': 'mean',
        'DaysOnMarket': 'mean'
    })
    .reset_index()
)

# Rename columns
county_summary.columns = [
    'CountyOrParish',
    'mean_list_price',
    'median_list_price',
    'avg_price_per_sqft',
    'avg_days_on_market'
]

county_summary = county_summary.sort_values(
    by='median_list_price',
    ascending=False
)

county_summary.head()

,CountyOrParish,mean_list_price,median_list_price,avg_price_per_sqft,avg_days_on_market
45,San Mateo,2.082118e+06,1598400.0,1044.541665,17.027038
47,Santa Clara,1.817617e+06,1498000.0,975.953077,15.688086
22,Marin,1.552133e+06,1285000.0,797.243477,23.915556
48,Santa Cruz,1.404366e+06,1200000.0,813.521154,23.521581
31,Orange,1.525801e+06,1199000.0,742.883591,17.959201


In [11]:
type_summary = listing.groupby('PropertyType').agg({
    'ListPrice': 'median',
    'price_per_sqft': 'mean',
    'DaysOnMarket': 'mean'
}).reset_index()

type_summary.columns = [
    'PropertyType',
    'median_list_price',
    'avg_price_per_sqft',
    'avg_days_on_market'
]

type_summary

,PropertyType,median_list_price,avg_price_per_sqft,avg_days_on_market
0,Residential,830000.0,599.801284,19.105181


In [14]:
listing.to_csv('../../IDX_Data/listing_week6_features.csv', index=False)
print('Listing Week 6 dataset saved.')

Listing Week 6 dataset saved.
